# Notebook 19 — regras de decisao sobre OOFs congeladas (sem treino)

**Contexto:** `docs/pesquisa_ganhos.md` (skill research) + `docs/diagnostico_erros.md`
(subagente). Alvo: teto honesto 0.504-0.509, erros 72% off-by-one, bolsos 11-14
(38%) e 4-7 (31%), par 9<->10 simetrico com classes boas (0.65).
**Protocolo:** so OOFs honestas (`nb11_oof311`, `nb12_e23_fh1s`, `nb16_honest_fold*`
variantes `orig_*`). Nada de relabel/quarentena (canal medido NB16). Regras com
parametros sao sintonizadas em folds internas e julgadas uma vez nas externas.
**B parqueado:** re-decodificacao rank-consistente exige as 15 cumulativas (so ha
probas finais nos npz) = 75 refits; so corre se A/C/D falharem.


In [ ]:
import os, json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
DATA_DIR = '../../data' if os.path.isdir('../../data') else ('../data' if os.path.isdir('../data') else 'data')
K, TARGET = 16, 'AGE_RANGE'
tr = pd.read_csv(os.path.join(DATA_DIR, 'train_nb02.csv'))
y = tr[TARGET].to_numpy().astype(int)
Xv = tr.drop(columns=['PID', TARGET]).to_numpy(dtype=np.float64)
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(Xv, y))
zx = np.load(os.path.join(DATA_DIR, 'nb11_oof311.npz'))
xgb_proba, xgb_oof = zx['proba'], zx['oof'].astype(int)
ze = np.load(os.path.join(DATA_DIR, 'nb12_e23_fh1s.npz'))
fh_proba = ze['proba']
fh = {f'orig_{m}': np.vstack([np.load(os.path.join(DATA_DIR, f'nb16_honest_fold{j}.npz'))[f'orig_{m}']
                              for j in range(5)]) for m in ['FH', 'mc5', 'LGB', 'HGB']}
print('xgb311 argmax: %.4f' % f1_score(y, xgb_oof, average='macro'))
print('fh1s argmax: %.4f' % f1_score(y, fh_proba.argmax(1), average='macro'))
RESULTADOS = []


## A — bias por classe ANINHADO (versao honesta da E15)

E15 sintonizou nas OOFs todas (+0.0019, otimista). Aqui cada fold e avaliado com
o bias sintonizado so nos outros 4 folds (coordinate ascent, passos 0.05/0.1).
Duas OOFs: XGB-311 e F&H-1seed.


In [ ]:
A = {}
for tag, proba in [('xgb311', xgb_proba), ('fh1s', fh_proba)]:
    logp = np.log(np.clip(proba, 1e-12, None))
    def _f1(b, idx):
        return f1_score(y[idx], (logp[idx] + b[None, :]).argmax(axis=1), average='macro')
    Bm = np.zeros((len(FOLDS), K))
    for fi, (trn, va) in enumerate(FOLDS):
        outros = np.concatenate([v for fj, (_, v) in enumerate(FOLDS) if fj != fi])
        b = np.zeros(K)
        base = _f1(b, outros)
        for _ in range(3):
            for k in range(K):
                for dd in (0.05, -0.05, 0.1, -0.1):
                    b2 = b.copy(); b2[k] += dd
                    f = _f1(b2, outros)
                    if f > base + 1e-6:
                        base, b = f, b2
        Bm[fi] = b
    pred = np.empty(len(y), dtype=int)
    for fi, (_, va) in enumerate(FOLDS):
        pred[va] = (logp[va] + Bm[fi][None, :]).argmax(axis=1)
    f1s = [float(f1_score(y[va], pred[va], average='macro')) for _, va in FOLDS]
    A[tag] = {'f1': f1s, 'B': Bm.tolist()}
    print('%s nested-bias: %.4f +/- %.4f' % (tag, np.mean(f1s), np.std(f1s)), flush=True)
    RESULTADOS.append({'exp': 'A', 'variante': 'bias aninhado ' + tag, 'f1': round(float(np.mean(f1s)), 4),
                       'std': round(float(np.std(f1s)), 4),
                       'nota': 'folds ganhos vs argmax: %d/5' % int((np.array(f1s) > np.array(
                           [f1_score(y[va], proba[va].argmax(1), average='macro') for _, va in FOLDS])).sum())})
np.savez(os.path.join(DATA_DIR, 'nb19_bias.npz'), **{'B_' + t: np.array(A[t]['B']) for t in A})
print('guardado nb19_bias.npz')


## C — suavizacao adjacente com peso sintonizado em folds internas

p_suave = (1-2a)*p + a*(vizinhos); a escolhido por fold externa nas outras 4.
Explora diretamente a massa 72% off-by-one. Duas OOFs.


In [ ]:
C = {}
def _smooth(p, a):
    s = np.zeros_like(p)
    s[:, 1:-1] = (1 - 2 * a) * p[:, 1:-1] + a * (p[:, :-2] + p[:, 2:])
    s[:, 0] = (1 - a) * p[:, 0] + a * p[:, 1]
    s[:, -1] = (1 - a) * p[:, -1] + a * p[:, -2]
    return s / s.sum(axis=1, keepdims=True)
for tag, proba in [('xgb311', xgb_proba), ('fh1s', fh_proba)]:
    GRID = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25]
    pred = np.empty(len(y), dtype=int)
    chosen = []
    for fi, (trn, va) in enumerate(FOLDS):
        outros = np.concatenate([v for fj, (_, v) in enumerate(FOLDS) if fj != fi])
        best_a, best_f = 0.0, -1.0
        for a in GRID:
            f = f1_score(y[outros], _smooth(proba[outros], a).argmax(1), average='macro')
            if f > best_f:
                best_a, best_f = a, f
        chosen.append(best_a)
        pred[va] = _smooth(proba[va], best_a).argmax(1)
    f1s = [float(f1_score(y[va], pred[va], average='macro')) for _, va in FOLDS]
    C[tag] = {'f1': f1s, 'alpha': chosen}
    print('%s smooth: %.4f +/- %.4f alphas=%s' % (tag, np.mean(f1s), np.std(f1s), chosen), flush=True)
    RESULTADOS.append({'exp': 'C', 'variante': 'suavizacao adjacente ' + tag, 'f1': round(float(np.mean(f1s)), 4),
                       'std': round(float(np.std(f1s)), 4), 'nota': 'alphas %s' % (chosen,)})


## D — regra mediana/esperanca sobre OOF congelada (diagnostico #3 do subagente)

Aplica decisao tipo F&H (mediana da distribuicao / valor esperado arredondado)
as probas XGB-311: o par 9<->10 simetrico encolhe? Sem retreino.


In [ ]:
D = {}
cum = np.cumsum(xgb_proba, axis=1)
pred_med = (cum < 0.5).sum(axis=1)
pred_exp = np.clip(np.round((xgb_proba * np.arange(K)[None, :]).sum(axis=1)), 0, K - 1).astype(int)
for nome, pred in [('mediana', pred_med), ('esperado', pred_exp)]:
    f1s = [float(f1_score(y[va], pred[va], average='macro')) for _, va in FOLDS]
    D[nome] = f1s
    print('%s: %.4f +/- %.4f' % (nome, np.mean(f1s), np.std(f1s)), flush=True)
    RESULTADOS.append({'exp': 'D', 'variante': 'XGB regra ' + nome, 'f1': round(float(np.mean(f1s)), 4),
                       'std': round(float(np.std(f1s)), 4), 'nota': ''})
c910 = ((y == 9) | (y == 10))
for nome, pred in [('argmax', xgb_oof), ('mediana', pred_med)]:
    n910 = int(((y[c910] == 9) & (pred[c910] == 10)).sum())
    n109 = int(((y[c910] == 10) & (pred[c910] == 9)).sum())
    print(nome, '9->10 / 10->9:', (n910, n109), flush=True)
pd.DataFrame(RESULTADOS).to_csv(os.path.join(DATA_DIR, 'nb19_resultados.csv'), index=False)
print('guardado data/nb19_resultados.csv')
